# Method or representation: what decides how close typical days come?

Suppose you want typical days that come as close as possible to the days they stand for, and you
compare clustering methods to find the best one. Two settings decide how close they come: the
clustering **method** decides which days share a typical day, and the **representation** decides
what that typical day is. Which of the two matters more?

In this tutorial we compare the three methods that group days by similarity, `hierarchical`,
`kmeans` and `kmedoids`, each with two representations, on less and more data. By the end you
will have seen that

* the three methods come within a few percent of each other,
* switching the representation from `medoid` to `mean` changes every method by far more, and
* this holds from two weeks of data to a full year.

The other methods aim at something else, so we leave them out: `averaging` and `contiguous` keep
the order of the calendar (see [Seasonal storage](clustering/02_seasonal_storage.ipynb)), and
`kmaxoids` gives the most unusual days typical days of their own (see
[Comparing clustering methods](comparing_clustering_methods.ipynb)).

## 1  Three methods, one representation

We start with six weeks of hourly data: solar irradiance, temperature, wind and load. Every run
reduces the data to typical days with the same settings, so we write them down once in a small
function; only the data, the method, the representation and the number of typical days change.
`preserve_column_means=False` switches off tsam's rescaling step.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

import tsam
from tsam import ClusterConfig
from tsam.plot import compare_partitions

pio.renderers.default = "notebook_connected"

raw = pd.read_csv("../data/testdata.csv", index_col=0, parse_dates=True)
weeks = raw.iloc[: 42 * 24]  # the first six weeks of hourly data
METHODS = ["hierarchical", "kmeans", "kmedoids"]


def aggregate(data, method, representation, n_typical=6):
    np.random.seed(0)  # kmeans starts from random centers
    return tsam.aggregate(
        data,
        n_clusters=n_typical,
        period_duration="1D",
        cluster=ClusterConfig(method=method, representation=representation),
        preserve_column_means=False,
        numerical_tolerance=1e-9,  # tolerate rounding noise in tsam's range check
    )

Run the three methods with `medoid`, which copies one real day of each group, and see which days
share a typical day. Each column is one of the 42 days in calendar order, each row a method, and
the colour is the group the day lands in:

In [ ]:
with_medoid = {method: aggregate(weeks, method, "medoid") for method in METHODS}
compare_partitions(with_medoid, show_ids=False, title="Which days share a typical day")

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
def canonical(assignments):
    first_seen = {}
    return [first_seen.setdefault(int(c), len(first_seen)) for c in assignments]


groupings = {m: canonical(r.cluster_assignments) for m, r in with_medoid.items()}
assert len(groupings["hierarchical"]) == 42
assert len({tuple(g) for g in groupings.values()}) == 3

The three methods group most days alike, but not all. How close do their typical days come?
`result.accuracy.weighted_rmse` measures how far the typical days are from the days they stand
for, hour by hour, in normalized units; lower is better.

In [ ]:
pd.Series(
    {method: result.accuracy.weighted_rmse for method, result in with_medoid.items()},
    name="RMSE with medoid",
).round(3)

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
medoid_rmse = pd.Series({m: r.accuracy.weighted_rmse for m, r in with_medoid.items()})
assert medoid_rmse.max() / medoid_rmse.min() < 1.02

Despite the different groups, all three come within 2% of each other.

## 2  The same methods, another representation

`mean` averages the days of each group, hour by hour. Run the three methods again with `mean`,
and put the result next to step 1:

In [ ]:
with_mean = {method: aggregate(weeks, method, "mean") for method in METHODS}
pd.DataFrame(
    {
        "medoid": {
            method: result.accuracy.weighted_rmse
            for method, result in with_medoid.items()
        },
        "mean": {
            method: result.accuracy.weighted_rmse
            for method, result in with_mean.items()
        },
    }
).round(3)

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
mean_rmse = pd.Series({m: r.accuracy.weighted_rmse for m, r in with_mean.items()})
assert mean_rmse.max() / mean_rmse.min() < 1.03
gain = medoid_rmse / mean_rmse - 1
assert gain.min() >= 0.15 and gain.max() < 0.205

With either representation, the three methods stay within 3% of each other. But with `medoid`,
every method is 15 to 20% further from the real days than with `mean`. The representation
decides more than the method.

Keep this in mind when you compare methods yourself: a bare `ClusterConfig(method=...)` picks
the representation for you, `mean` for `kmeans` and `medoid` for `hierarchical` and `kmedoids`.
Comparing methods with their defaults compares representations too.

`medoid` still has its uses: its typical days are real days, which some models need.
[Comparing representations](comparing_representations.ipynb) shows what each representation
keeps besides the fit.

## 3  Less and more data

Does this hold for less and more data? Repeat both comparisons on two weeks, six weeks, a quarter,
half a year and a full year, still with 6 typical days. `kmedoids` solves an optimization problem
whose runtime grows quickly with the data (see
[As similar as possible](clustering/01_similar_days.ipynb)), so here it runs only up to a quarter.

In [ ]:
def compare_amounts(n_typical):
    rows = []
    for days in [14, 42, 91, 182, 365]:
        data = raw.iloc[: days * 24]
        for method in METHODS:
            if method == "kmedoids" and days > 91:
                continue
            for representation in ["medoid", "mean"]:
                result = aggregate(data, method, representation, n_typical)
                rows.append(
                    {
                        "days": days,
                        "method": method,
                        "representation": representation,
                        "RMSE": result.accuracy.weighted_rmse,
                    }
                )
    return pd.DataFrame(rows)


def plot_amounts(scores, title):
    fig = px.line(
        scores,
        x="days",
        y="RMSE",
        color="method",
        line_dash="representation",
        line_dash_map={"medoid": "dash", "mean": "solid"},
        markers=True,
        title=title,
    )
    fig.update_xaxes(
        type="log", tickvals=sorted(scores["days"].unique()), title_text="days of data"
    )
    return fig


scores = compare_amounts(6)
plot_amounts(scores, "6 typical days: RMSE by amount of data")

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
wide = scores.pivot_table(
    index=["days", "method"], columns="representation", values="RMSE"
)
representation_gain = wide["medoid"] / wide["mean"] - 1
method_spread = (
    wide.groupby(level="days").agg(lambda s: s.max() / s.min() - 1).max(axis=1)
)
assert set(scores["days"]) == {14, 42, 91, 182, 365}
assert (
    round(representation_gain.min(), 2) == 0.11
    and round(representation_gain.max(), 2) == 0.35
)
assert method_spread.max() < 0.07
assert (representation_gain.groupby(level="days").min() > method_spread).all()
hierarchical_gain = representation_gain.xs("hierarchical", level="method")
assert (
    hierarchical_gain.is_monotonic_decreasing
    or hierarchical_gain.iloc[0] > hierarchical_gain.iloc[-1]
)
assert (
    round(hierarchical_gain[14], 2) == 0.35 and round(hierarchical_gain[365], 2) == 0.14
)

The dashed `medoid` lines run above the solid `mean` lines at every amount of data, and within
each representation the methods stay close together. For every method and every amount of data,
switching from `medoid` to `mean` changes the RMSE by 11 to 35%, while the methods differ by at
most 7%. The gap between the representations narrows as the data grows, for `hierarchical` from
35% on two weeks to 14% on a full year, but it stays larger than the gap between the methods.

## 4  Try it yourself

Change the number of typical days and run the comparison again. With more typical days, each
group gets smaller; does the representation still matter more than the method?

In [ ]:
N_TYPICAL = 12  # try 4, 8 or 20
plot_amounts(
    compare_amounts(N_TYPICAL), f"{N_TYPICAL} typical days: RMSE by amount of data"
)

## What you've learned

* `hierarchical`, `kmeans` and `kmedoids` group some days differently, but their typical days come
  within a few percent of each other.
* The representation moves the fit far more: with `medoid`, every method is 11 to 35% further from
  the real days than with `mean`.
* This holds from two weeks of data to a full year. The gap between the representations narrows
  with more data, but stays larger than the gap between the methods.
* Compare methods with the same representation: a bare `ClusterConfig(method=...)` gives them
  different ones.

Where to go next:

* [1. As similar as possible](clustering/01_similar_days.ipynb) — what the three methods cost in
  runtime.
* [2. Seasonal storage](clustering/02_seasonal_storage.ipynb) — what `averaging` and `contiguous`
  are for.
* [Comparing representations](comparing_representations.ipynb) — what each representation keeps
  besides the fit.
* [Clustering methods](../how-to/clustering_methods.ipynb) — pick a method by what your model
  needs.